# Star Schema Exploratory Data Analysis (EDA)

This notebook queries the `datamart.duckdb` built by the `DataMart-Flex` ETL pipeline to validate relationships and compute baseline DAX equivalent measures in Python/SQL.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to the data mart
con = duckdb.connect('../data/datamart.duckdb')


## 1. Table Verification

In [ ]:
tables = con.execute("SHOW TABLES").df()
print(tables)

## 2. Emulating DAX Measures via SQL
Calculating *Total Revenue*, *Gross Margin*, and grouping by *Channel* (simulating a BI Matrix visualization).

In [ ]:
query = """
SELECT 
    c.channel_name,
    COUNT(DISTINCT f.order_id) AS total_orders,
    SUM(f.sales_amount) AS total_revenue,
    SUM(f.sales_amount - f.cost_amount) AS gross_margin_usd,
    (SUM(f.sales_amount - f.cost_amount) / SUM(f.sales_amount)) * 100 AS gross_margin_pct
FROM Fact_Orders f
JOIN Dim_Channels c ON f.channel_key = c.channel_key
GROUP BY c.channel_name
ORDER BY total_revenue DESC;
"""
df_channel_perf = con.execute(query).df()
display(df_channel_perf)

## 3. Time Series Visualization (Monthly Revenue Growth)
Connecting `Fact_Orders` to `Dim_Date`.

In [ ]:
query_ts = """
SELECT 
    d.year,
    d.month_num,
    d.month_name,
    SUM(f.sales_amount) AS monthly_revenue
FROM Fact_Orders f
JOIN Dim_Date d ON f.date_key = d.date_key
GROUP BY d.year, d.month_num, d.month_name
ORDER BY d.year, d.month_num;
"""
df_ts = con.execute(query_ts).df()
df_ts['YearMonth'] = df_ts['year'].astype(str) + '-' + df_ts['month_num'].astype(str).str.zfill(2)

plt.figure(figsize=(14,6))
sns.lineplot(data=df_ts, x='YearMonth', y='monthly_revenue', marker='o')
plt.xticks(rotation=45)
plt.title('Monthly Revenue Trend')
plt.ylabel('Total Revenue ($)')
plt.xlabel('Month')
plt.tight_layout()
plt.show()